# Examen No. 3 – Aprendizaje Supervisado
## Comparación de Algoritmos de Clasificación
### Universidad Pontificia Bolivariana
**Curso:** Inteligencia Artificial
**Docente:** Juan Darío Rodas  
**Dataset:** Palmer Penguins (UCI ML Repository)  
**Algoritmos:** Logistic Regression · Random Forest · Support Vector Machine (SVM)

---
## 1. Importación de Librerías

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.ticker as mticker

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

SEED = 42
sns.set_theme(style='whitegrid', palette='Set2', font_scale=1.15)
plt.rcParams['figure.dpi'] = 110


---
## 2. Carga y Exploración Inicial del Dataset

In [ ]:
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00690/palmer_penguins.csv'

try:
    df = pd.read_csv(url)
except Exception:
    !pip install palmerpenguins -q
    from palmerpenguins import load_penguins
    df = load_penguins()

df.columns = [
    c.strip().lower()
    .replace('culmen_length_mm', 'bill_length_mm')
    .replace('culmen_depth_mm', 'bill_depth_mm')
    for c in df.columns
]

print('filas, columnas:', df.shape)
df.head()


In [ ]:
if df['species'].dtype == object and df['species'].str.contains('\\d', na=False).any():
    df['species'] = df['species'].str.extract(r'([A-Za-z]+)')[0]

df.columns.tolist()


In [ ]:
try:
    from palmerpenguins import load_penguins
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'palmerpenguins', '-q'])
    from palmerpenguins import load_penguins

df = load_penguins()
print(df.shape, df.columns.tolist())
df.head()


In [ ]:
df.info()


In [ ]:
df.describe(include='all').round(2)


In [ ]:
nulos = df.isnull().sum()
pct = (df.isnull().mean() * 100).round(2)
pd.DataFrame({'count': nulos, 'pct': pct})[nulos > 0]


In [ ]:
df['species'].value_counts()


---
## 3. EDA – Análisis Exploratorio y Calidad de Datos

### 3.1 Distribución de Clases

In [ ]:
conteo = df['species'].value_counts().reset_index()
conteo.columns = ['species', 'n']

fig, ax = plt.subplots(figsize=(7, 5))
colores = sns.color_palette('Set2', 3)
barras = ax.bar(conteo['species'], conteo['n'], color=colores, edgecolor='white', width=0.55)

for b, v in zip(barras, conteo['n']):
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 2,
            str(v), ha='center', va='bottom', fontweight='bold')

ax.set_title('Distribución de especies')
ax.set_xlabel('Especie')
ax.set_ylabel('Frecuencia')
ax.set_ylim(0, conteo['n'].max() * 1.15)
sns.despine()
plt.tight_layout()
plt.show()


### 3.2 Distribución de Variables Numéricas por Especie (Boxplots)

In [ ]:
variables = [
    ('bill_length_mm', 'Longitud pico (mm)'),
    ('bill_depth_mm', 'Profundidad pico (mm)'),
    ('flipper_length_mm', 'Longitud aleta (mm)'),
    ('body_mass_g', 'Masa corporal (g)'),
]

colores_esp = {'Adelie': '#66C2A5', 'Chinstrap': '#FC8D62', 'Gentoo': '#8DA0CB'}

for col, titulo in variables:
    fig, ax = plt.subplots(figsize=(7, 5))
    sns.boxplot(data=df, x='species', y=col, palette=colores_esp, width=0.5, ax=ax)
    ax.set_title(titulo)
    ax.set_xlabel('Especie')
    ax.set_ylabel(titulo)
    sns.despine()
    plt.tight_layout()
    plt.show()


### 3.3 Matriz de Correlación (Heatmap)

In [ ]:
cols_num = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']
corr = df[cols_num].corr()

fig, ax = plt.subplots(figsize=(7, 5.5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            vmin=-1, vmax=1, linewidths=0.5, linecolor='white', ax=ax)
ax.set_title('Correlación entre variables numéricas')
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
plt.tight_layout()
plt.show()


---
## 4. Preprocesamiento de Datos

### 4.1 Selección de Características y Variable Objetivo

In [ ]:
target = 'species'
features = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g', 'island', 'sex']
num_feat = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']
cat_feat = ['island', 'sex']

X = df[features].copy()
y = df[target].copy()

print(X.shape, y.shape)
y.value_counts()


### 4.2 División Train / Test (80 % – 20 %, random_state=42)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

print(f'train: {len(X_train)}  test: {len(X_test)}')
print(y_train.value_counts())


### 4.3 Pipeline de Preprocesamiento (imputación → codificación → normalización)

In [ ]:
from sklearn.preprocessing import OneHotEncoder

pipe_num = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

pipe_cat = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ohe', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'))
])

prep = ColumnTransformer([
    ('num', pipe_num, num_feat),
    ('cat', pipe_cat, cat_feat)
])

X_train_t = prep.fit_transform(X_train)
X_test_t = prep.transform(X_test)

print(X_train_t.shape, X_test_t.shape)


---
## 5. Entrenamiento de Modelos

In [ ]:
clfs = {
    'Logistic Regression': LogisticRegression(max_iter=1000, C=1.0, solver='lbfgs',
                                               multi_class='multinomial', random_state=SEED),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1),
    'SVM': SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=SEED)
}

for nombre, clf in clfs.items():
    clf.fit(X_train_t, y_train)
    print(f'{nombre} listo')


---
## 6. Evaluación de Métricas

In [ ]:
resultados = {}

for nombre, clf in clfs.items():
    pred = clf.predict(X_test_t)
    resultados[nombre] = {
        'Accuracy': round(accuracy_score(y_test, pred), 4),
        'Precision': round(precision_score(y_test, pred, average='weighted', zero_division=0), 4),
        'Recall': round(recall_score(y_test, pred, average='weighted', zero_division=0), 4),
        'F1': round(f1_score(y_test, pred, average='weighted', zero_division=0), 4),
        'pred': pred
    }

tabla = pd.DataFrame(resultados).T.drop(columns='pred').astype(float)
display(tabla.style.format('{:.4f}').highlight_max(color='#b7e4c7', axis=0))


In [ ]:
for nombre in clfs:
    print(f'\n{nombre}')
    print(classification_report(y_test, resultados[nombre]['pred'], zero_division=0))


---
## 7. Matrices de Confusión

In [ ]:
etiquetas = sorted(y_test.unique())
cmaps = ['Blues', 'Greens', 'Oranges']

for (nombre, clf), cm_color in zip(clfs.items(), cmaps):
    pred = resultados[nombre]['pred']
    cm = confusion_matrix(y_test, pred, labels=etiquetas)
    fig, ax = plt.subplots(figsize=(6, 5))
    ConfusionMatrixDisplay(cm, display_labels=etiquetas).plot(ax=ax, cmap=cm_color, values_format='d')
    ax.set_title(f'Confusión – {nombre}')
    plt.tight_layout()
    plt.show()


---
## 8. Comparación Visual de Métricas

In [ ]:
metricas = ['Accuracy', 'Precision', 'Recall', 'F1']
algos = list(tabla.index)
x = np.arange(len(algos))
w = 0.18
pal = sns.color_palette('Set2', len(metricas))

fig, ax = plt.subplots(figsize=(9, 5.5))
for i, (m, c) in enumerate(zip(metricas, pal)):
    vals = [resultados[a][m] for a in algos]
    offset = (i - len(metricas)/2 + 0.5) * w
    bars = ax.bar(x + offset, vals, width=w, label=m, color=c, edgecolor='white')
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.005,
                f'{v:.3f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(algos)
ax.set_ylim(0, 1.12)
ax.set_xlabel('Algoritmo')
ax.set_ylabel('Métrica')
ax.set_title('Comparación de métricas por algoritmo')
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left')
sns.despine()
plt.tight_layout()
plt.show()


---
## 9. Selección del Mejor Modelo

In [ ]:
display(tabla.style.format('{:.4f}').highlight_max(color='#b7e4c7', axis=0))

for m, best in tabla.idxmax().items():
    print(f'{m}: {best} ({tabla.loc[best, m]:.4f})')


---
## 10. Preguntas de Análisis e Interpretación

### Pregunta 1
**El dataset Pingüinos del archipiélago de Palmer contiene variables numéricas (bill_length_mm, body_mass_g, etc.) y categóricas (island, sex). ¿Por qué es necesario aplicar encoding y normalización antes de entrenar cada uno de los tres modelos? ¿Todos los algoritmos los requieren con la misma urgencia?**

---

Los modelos de machine learning trabajan exclusivamente con números. Columnas como `island` o `sex` contienen texto, así que si no se transforman, los algoritmos directamente no pueden procesarlas. El One-Hot Encoding convierte cada categoría en una columna binaria (0 o 1), sin asumir ningún orden entre los valores.

La normalización resuelve otro problema: las variables numéricas del dataset tienen unidades completamente distintas. `body_mass_g` puede llegar a 6000 mientras que `bill_depth_mm` ronda 15-20. Sin estandarizar, esa diferencia de magnitud haría que algunas variables dominen los cálculos simplemente por ser más grandes en número, no porque sean más informativas.

Dicho esto, no todos los modelos tienen la misma dependencia:

- **SVM con kernel RBF**: es el más sensible. Calcula distancias entre puntos para construir el margen, así que si las escalas son muy distintas, las variables grandes distorsionan todo. Sin normalización su rendimiento puede caer significativamente.
- **Regresión Logística**: también depende de la escala porque optimiza pesos mediante gradiente. Variables en escalas distintas hacen que el proceso de optimización sea más lento e inestable.
- **Random Forest**: técnicamente no necesita normalización, ya que los árboles funcionan por umbrales en una variable a la vez. Pero el encoding sigue siendo obligatorio.

Resumiendo: el encoding es necesario para los tres. La normalización es crítica para SVM y Logistic Regression, pero opcional para Random Forest.


### Pregunta 2
**Antes de entrenar cualquier modelo, el dataset presenta aproximadamente 19 valores faltantes. ¿Cuál fue la estrategia de imputación que utilizaste y cómo una mala decisión en este paso podría afectar el desempeño de los tres clasificadores?**

---

Para las variables numéricas usé imputación con la **mediana**, y para las categóricas con la **moda**. Todo dentro de un `Pipeline` de sklearn para que los estadísticos se calcules únicamente sobre el conjunto de entrenamiento y luego se apliquen al de prueba.

Elegí la mediana en lugar de la media porque en los boxplots se observaron valores atípicos en `body_mass_g` y `bill_length_mm`. La media se ve afectada por esos extremos, mientras que la mediana no.

**Qué podría salir mal con otras decisiones:**

Eliminar directamente las filas con NaN suena simple pero con 344 registros totales, perder 19 no es trivial: representa un 5.5% del dataset y puede desbalancear aún más las clases. Los modelos quedarían entrenados con menos información de la que deberían.

Imputar con la media cuando hay outliers introduce un valor que probablemente no represente bien la distribución real. Eso puede desplazar las fronteras de decisión de Logistic Regression y SVM, que son sensibles a la distribución de los datos.

El error más grave sería calcular la mediana sobre todo el dataset (incluyendo test) antes de dividirlo. Eso es data leakage: el modelo vería indirectamente información del conjunto de prueba durante el entrenamiento, lo que infla artificialmente las métricas y no refleja el rendimiento real en datos nuevos.


### Pregunta 3
**La clase Adelie representa cerca del 44 % de los datos, mientras que Chinstrap es la menos representada. ¿Esto constituye un problema de desbalance de clases? ¿Qué impacto podría tener sobre cada una de las métricas?**

---

El desbalance en este dataset es leve. La distribución aproximada es Adelie ~44%, Gentoo ~36%, Chinstrap ~20%. En términos generales, un desbalance se vuelve problemático cuando una clase tiene menos del 5-10% del total. Con 20% Chinstrap sigue siendo una minoría a considerar, pero no es un caso extremo.

El problema real es que las métricas no lo cuentan todo igual:

**Accuracy**: puede ser engañosa. Si el modelo aprendiera a predecir siempre Adelie, acertaría en casi la mitad de los casos sin haber aprendido nada útil. Parece buen resultado pero no lo es.

**Precision y Recall por clase**: para Chinstrap estos valores van a ser más bajos que para las otras dos especies, simplemente porque hay menos ejemplos para aprender sus patrones. El modelo tiene mayor tendencia a clasificarla como Adelie.

**F1-Score**: al ser el promedio armónico de precisión y recall, penaliza más cuando uno de los dos cae. Por eso es la métrica más honesta cuando hay desbalance, ya que no se deja llevar solo por la clase mayoritaria.

Para mitigar este efecto, se podría usar `class_weight='balanced'` en Logistic Regression y SVM, que ajusta los pesos de cada clase inversamente proporcional a su frecuencia.


### Pregunta 4
**La Regresión Logística asume una relación lineal entre las variables de entrada y la frontera de decisión. Dado que las especies Adelie y Chinstrap se solapan en el espacio de características, ¿crees que este algoritmo tendrá dificultades para separarlas? ¿Qué estrategia podría mejorar su desempeño?**

---

Sí, la Regresión Logística va a tener problemas con Adelie y Chinstrap. Mirando los boxplots, estas dos especies se solapan bastante en `bill_length_mm` y `bill_depth_mm`. La Regresión Logística busca un hiperplano que divida las clases, y cuando el solapamiento es real no existe un plano que las separe limpiamente.

Eso no significa que el modelo falle completamente: con variables adicionales como `flipper_length_mm` y `body_mass_g` puede construir una frontera razonable en el espacio multivariado. Pero en comparación con Random Forest o SVM, tiene una desventaja estructural para este par de clases.

**Formas de mejorar su desempeño:**

La opción más directa sería agregar `PolynomialFeatures` antes del modelo para generar términos cuadráticos e interacciones entre variables. Así la frontera sigue siendo lineal en el espacio transformado pero no lineal en el espacio original.

Otra opción es ajustar el parámetro `C` mediante `GridSearchCV`. Un `C` más alto reduce la regularización y permite que el modelo se ajuste más a los datos; un `C` más bajo generaliza más. Encontrar el valor óptimo puede mejorar el rendimiento sobre las clases difíciles.

También se podría probar LDA (Linear Discriminant Analysis) como paso previo, ya que maximiza explícitamente la separación entre clases antes de clasificar.


### Pregunta 5
**Tras entrenar los tres modelos, se obtienen las matrices de confusión. ¿Cuál de los tres comete errores más críticos al confundir especies muy distintas morfológicamente y cómo lo identificarías en la matriz?**

---

Primero hay que definir qué hace a un error más crítico que otro. Confundir Adelie con Chinstrap es relativamente menos grave porque morfológicamente son parecidas: tamaño similar, hábitat compartido. Confundir cualquiera de ellas con Gentoo sería el error más crítico, porque Gentoo es una especie notablemente diferente: más grande, con aletas más largas y una coloración distinta.

En la matriz de confusión, los errores aparecen en las celdas fuera de la diagonal. Si hay un valor alto en la intersección fila-Gentoo / columna-Adelie (o viceversa), significa que el modelo está confundiendo dos especies que claramente no deberían confundirse.

La Regresión Logística es el candidato más probable a cometer errores entre Adelie y Chinstrap, por la razón mencionada en la pregunta anterior: su frontera lineal no puede separar bien clases con solapamiento. Random Forest y SVM generalmente manejan mejor este par.

Si alguno de los modelos confunde Gentoo con otra especie, eso indica un fallo serio. Gentoo es tan diferente en las variables numéricas que separarla del resto debería ser relativamente sencillo para cualquiera de los tres algoritmos.


### Pregunta 6
**El dataset fue recolectado en tres islas distintas (Biscoe, Dream, Torgersen), y ciertas especies solo habitan en algunas de ellas. Si incluyes island como variable predictora, ¿el modelo estaría aprendiendo biología real o simplemente memorizando una regla geográfica? ¿Cómo afectaría esto a la capacidad de generalización del clasificador en nuevas ubicaciones?**

---

Mayormente estaría aprendiendo una regla geográfica. En este dataset:
- Torgersen tiene solo Adelie
- Biscoe tiene principalmente Gentoo
- Dream tiene mezcla de Adelie y Chinstrap

Esa distribución hace que `island` sea un predictor muy potente para este dataset específico, pero por razones que no tienen nada que ver con la biología del animal. El modelo aprendería básicamente que "si la isla es X, entonces la especie es Y", que es una correlación válida aquí pero completamente frágil.

El problema aparece cuando se intenta usar el modelo fuera de este contexto: una nueva colonia de Chinstrap en Torgersen, datos de otra región del Antártico, o simplemente individuos cuya isla de origen no se conoce. En todos esos casos el modelo fallaría porque su "aprendizaje" estaba atado a la geografía, no a las características del pingüino.

Incluir `island` puede mejorar las métricas en este examen, pero para un clasificador que deba funcionar en condiciones reales o generalizarse a nuevos datos, es mejor apoyarse en variables morfológicas como `bill_length_mm`, `flipper_length_mm` o `body_mass_g`, que describen al individuo independientemente de dónde fue encontrado.


### Pregunta 7
**Considera la situación en la cual una sola variable resulte ser la más importante para la clasificación, por ejemplo flipper_length_mm. ¿Qué significaría esto en términos biológicos? ¿Podría un modelo entrenado solo con esa variable superar a la Regresión Logística entrenada con todas las variables?**

---

Si `flipper_length_mm` fuera la variable más importante, significaría que el tamaño de la aleta contiene suficiente información por sí solo para distinguir las especies. Biológicamente tiene sentido: Gentoo tiene las aletas más largas (~217 mm promedio), adaptadas a buceos profundos, mientras que Adelie y Chinstrap tienen aletas más cortas y similares entre sí. Una variable que separa tan bien a Gentoo del resto ya resuelve dos tercios del problema de clasificación.

El hecho de que Adelie y Chinstrap sean parecidas en `flipper_length_mm` también explicaría por qué la Regresión Logística tiene dificultades con ese par: si la variable más discriminativa tampoco las separa bien, el modelo tiene poco margen.

**¿Podría un modelo univariado superar a la Regresión Logística completa?**

En teoría sí. Si las otras variables introducen ruido o correlaciones que confunden al modelo, usar solo `flipper_length_mm` puede dar un resultado más limpio. Esto ocurre especialmente cuando hay multicolinealidad: `flipper_length_mm` y `body_mass_g` están muy correlacionadas (se ve en el heatmap), así que incluir ambas no necesariamente suma información independiente.

En la práctica, la Regresión Logística tiene regularización que reduce el peso de variables poco útiles, así que agregar más variables no siempre la perjudica. Pero sería un experimento válido entrenar un modelo solo con `flipper_length_mm` y comparar sus métricas para verificarlo.


---
## Conclusión General

Los tres algoritmos logran resultados sólidos en este dataset, pero con diferencias que vale la pena destacar.

Random Forest y SVM tienden a superar a la Regresión Logística principalmente en la separación entre Adelie y Chinstrap, donde la frontera lineal del modelo logístico es una limitación real. Ambos modelos no lineales manejan mejor ese solapamiento.

Dicho esto, la Regresión Logística no es despreciable: es mucho más interpretable y rápida de entrenar, y en un dataset de este tamaño la diferencia de rendimiento no es enorme.

Para elegir el mejor modelo en este contexto, el F1-score ponderado es la métrica más apropiada, ya que tiene en cuenta el leve desbalance entre clases y no se deja llevar solo por la clase mayoritaria. Si además se revisa la matriz de confusión y ningún modelo confunde Gentoo con las otras especies (lo que sería un error grave), entonces el criterio de selección se reduce a cuál tiene mejor F1 sobre Chinstrap específicamente.

El pipeline implementado con `fit_transform` en train y `transform` en test garantiza que no haya data leakage en ninguna etapa.


